# BeeMonitor System Test
Complete pipeline testing: Detection → Tracking → Events → Output

## Overview
This notebook tests the complete BeeMonitor system:
- **Phase 1**: Detector initialization (DL-verified background, SIFT templates)
- **Phase 2**: Tracking (multi-modal detection + MOT)
- **Phase 3**: Event analysis (entry/exit detection)
- **Phase 4**: Visualization and statistics

## Setup

In [1]:
import sys
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Ellipse, Circle
import seaborn as sns
from IPython.display import Video, display, HTML
from tqdm.notebook import tqdm

# Add beemonitor to path if needed
# sys.path.insert(0, '/path/to/beemonitor')

from ultralytics import YOLO

# BeeMonitor imports
from beemonitor.detection import (
    BlobDetector, SIFTDetector, YOLODetector, NestDetector
)
# Import NoiseFilter - try main module first, then specific file
try:
    from beemonitor.detection import NoiseFilter
except ImportError:
    try:
        from beemonitor.detection.noise_filter import NoiseFilter
    except ImportError:
        NoiseFilter = None
        print("Warning: NoiseFilter not available")

from beemonitor.tracking import BeeTracking, DetectionMode
from beemonitor.tracking.mot import BeeTracker
from beemonitor.core.config import BeeMonitorConfig

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Imports successful")

ImportError: cannot import name 'BeeMonitorConfig' from 'beemonitor.core.config' (/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/src/beemonitor/core/config.py)

## Configuration

In [ ]:
# Video configuration
VIDEO_PATH = 'path/to/bee_hotel.mp4'  # UPDATE THIS
# ROI will be determined automatically from nest detector

# Output paths
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

TEMPLATES_PATH = OUTPUT_DIR / 'bee_templates.pkl'
TRACKING_CSV = OUTPUT_DIR / 'tracking_results.csv'
EVENTS_CSV = OUTPUT_DIR / 'events.csv'
VIZ_VIDEO = OUTPUT_DIR / 'tracking_visualization.mp4'

# Model paths
YOLO_MODEL = 'yolo11n.pt'  # Or path to custom trained model
CNN_MODEL = 'bee_classifier.pth'  # Or None to skip noise filtering

# Detection configuration
DETECTION_MODE = DetectionMode.FGBG_SIFT_YOLO  # Most comprehensive
USE_NOISE_FILTER = True if CNN_MODEL else False

# ROI padding (pixels to add around detected hotel region)
ROI_PADDING = 20

# System configuration
config = BeeMonitorConfig()

print(f"Video: {VIDEO_PATH}")
print(f"Detection mode: {DETECTION_MODE.value}")
print(f"Noise filter: {USE_NOISE_FILTER}")
print(f"ROI padding: {ROI_PADDING}px")

## Utility Functions

In [ ]:
def show_frame(frame, title="Frame", figsize=(12, 8)):
    """Display a frame."""
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def draw_detections(frame, detections, color=(0, 255, 0)):
    """Draw detections on frame."""
    frame = frame.copy()
    for det in detections:
        x1, y1, x2, y2 = [int(c) for c in det.bbox]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        
        # Add label
        label = f"{det.label} ({det.source})"
        cv2.putText(frame, label, (x1, y1-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return frame

def draw_tracks(frame, tracks, show_velocity=True):
    """Draw tracks on frame."""
    frame = frame.copy()
    colors = plt.cm.tab20(np.linspace(0, 1, 20))
    
    for track_id, track in tracks.items():
        color = tuple(int(c * 255) for c in colors[track_id % 20][:3])
        
        # Draw bbox
        x1, y1, x2, y2 = [int(c) for c in track.bbox]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        
        # Draw ID
        cv2.putText(frame, f"ID:{track_id}", (x1, y1-10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
        # Draw velocity arrow
        if show_velocity and track.velocity:
            cx, cy = track.centroid
            vx, vy = track.velocity
            scale = 10
            end_x = int(cx + vx * scale)
            end_y = int(cy + vy * scale)
            cv2.arrowedLine(frame, (int(cx), int(cy)), (end_x, end_y),
                          color, 2, tipLength=0.3)
    
    return frame

def plot_tracking_stats(results_df):
    """Plot tracking statistics."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Tracks per frame
    tracks_per_frame = results_df.groupby('frame')['track_id'].nunique()
    axes[0, 0].plot(tracks_per_frame.index, tracks_per_frame.values)
    axes[0, 0].set_xlabel('Frame')
    axes[0, 0].set_ylabel('Number of Tracks')
    axes[0, 0].set_title('Active Tracks per Frame')
    axes[0, 0].grid(True)
    
    # 2. Track lengths
    track_lengths = results_df.groupby('track_id').size()
    axes[0, 1].hist(track_lengths, bins=30, edgecolor='black')
    axes[0, 1].set_xlabel('Track Length (frames)')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title('Distribution of Track Lengths')
    axes[0, 1].grid(True)
    
    # 3. Spatial distribution
    axes[1, 0].scatter(results_df['x1'], results_df['y1'], alpha=0.1, s=1)
    axes[1, 0].set_xlabel('X coordinate')
    axes[1, 0].set_ylabel('Y coordinate')
    axes[1, 0].set_title('Spatial Distribution of Detections')
    axes[1, 0].invert_yaxis()
    
    # 4. Species distribution (if available)
    if 'species' in results_df.columns:
        species_counts = results_df['species'].value_counts()
        axes[1, 1].bar(species_counts.index, species_counts.values)
        axes[1, 1].set_xlabel('Species')
        axes[1, 1].set_ylabel('Detections')
        axes[1, 1].set_title('Species Distribution')
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()

print("✓ Utility functions loaded")

---
# Phase 1: Detector Initialization

Initialize all detectors with DL verification and guidance.

## 1.1: Load Video and Detect Bee Hotel Region

In [ ]:
# Load video
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise ValueError(f"Cannot open video: {VIDEO_PATH}")

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video Properties:")
print(f"  Resolution: {width}x{height}")
print(f"  FPS: {fps}")
print(f"  Total frames: {total_frames}")
print(f"  Duration: {total_frames/fps:.1f}s")

# Read first frame
ret, first_frame = cap.read()
cap.release()

if not ret:
    raise ValueError("Failed to read first frame")

print("\n✓ First frame loaded")

## 1.2: Detect Bee Hotel and Calculate ROI

In [ ]:
print("Detecting bee hotel region...\n")

# Initialize nest detector
nest_detector = NestDetector()

# Detect nests
nests = nest_detector.detect_nests(first_frame)
print(f"✓ Detected {len(nests)} nests")

if len(nests) == 0:
    print("\n⚠ Warning: No nests detected!")
    print("  Using full frame as ROI")
    ROI = (0, 0, width, height)
else:
    # Calculate bounding box around all nests
    all_x1 = [nest.bbox[0] for nest in nests]
    all_y1 = [nest.bbox[1] for nest in nests]
    all_x2 = [nest.bbox[2] for nest in nests]
    all_y2 = [nest.bbox[3] for nest in nests]
    
    # ROI is bounding box of all nests + padding
    roi_x1 = max(0, int(min(all_x1)) - ROI_PADDING)
    roi_y1 = max(0, int(min(all_y1)) - ROI_PADDING)
    roi_x2 = min(width, int(max(all_x2)) + ROI_PADDING)
    roi_y2 = min(height, int(max(all_y2)) + ROI_PADDING)
    
    ROI = (roi_x1, roi_y1, roi_x2, roi_y2)
    
    roi_width = roi_x2 - roi_x1
    roi_height = roi_y2 - roi_y1
    
    print(f"\n✓ ROI calculated from nest detections:")
    print(f"  Position: ({roi_x1}, {roi_y1}) to ({roi_x2}, {roi_y2})")
    print(f"  Size: {roi_width}x{roi_height} pixels")
    print(f"  Coverage: {(roi_width * roi_height) / (width * height) * 100:.1f}% of frame")

# Visualize nests and ROI
frame_viz = first_frame.copy()

# Draw individual nests
for i, nest in enumerate(nests):
    x1, y1, x2, y2 = [int(c) for c in nest.bbox]
    cv2.rectangle(frame_viz, (x1, y1), (x2, y2), (0, 255, 255), 2)
    cv2.putText(frame_viz, f"N{i+1}", (x1+5, y1+20),
               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

# Draw ROI
x1, y1, x2, y2 = ROI
cv2.rectangle(frame_viz, (x1, y1), (x2, y2), (0, 255, 0), 3)
cv2.putText(frame_viz, "ROI", (x1+10, y1+40),
           cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

show_frame(frame_viz, f"Detected Bee Hotel (n={len(nests)} nests) + ROI")

# Show nest grid if available
if hasattr(nest_detector, 'get_nest_grid'):
    try:
        grid = nest_detector.get_nest_grid()
        if grid:
            print(f"\nNest Grid Layout:")
            print(f"  Rows: {grid.get('rows', 'N/A')}")
            print(f"  Columns: {grid.get('cols', 'N/A')}")
    except:
        pass

print(f"\n✓ Bee hotel detection complete")

## 1.3: Initialize YOLO Detector

In [ ]:
print("Initializing YOLO detector...")

# Load YOLO model
yolo_model = YOLO(YOLO_MODEL)

# Create detector
yolo = YOLODetector(
    model=yolo_model,
    tracking_classes=['bee', 'wasp'],
    conf_threshold=0.25,
    imgsz=320  # Fast mode
)

print(f"✓ YOLO detector ready")
print(f"  Model: {YOLO_MODEL}")
print(f"  Device: {yolo.device}")
print(f"  Image size: {320}")

# Test on first frame
print("\nTesting YOLO on first frame...")
yolo_dets = yolo.detect(first_frame)
print(f"  Detections: {len(yolo_dets)}")

if yolo_dets:
    frame_with_yolo = draw_detections(first_frame, yolo_dets, color=(255, 0, 0))
    show_frame(frame_with_yolo, f"YOLO Detections (n={len(yolo_dets)})")
    
    # Show detection details
    for i, det in enumerate(yolo_dets[:5]):  # First 5
        print(f"  Det {i}: {det.label} @ {det.centroid} (conf={det.confidence:.2f})")

## 1.4: Initialize Blob Detector (DL-Verified)

In [ ]:
print("Initializing Blob Detector (DL-verified background)...")
print("This searches for bee-free frames using YOLO verification.")
print("May take a few minutes...\n")

blob = BlobDetector(
    min_area=50.0,
    min_solidity=0.5
)

# DL-verified initialization
num_clean = blob.initialize_from_video_with_verification(
    video_path=VIDEO_PATH,
    yolo_detector=yolo,
    num_frames=100,  # Target 100 clean frames
    start_frame=0,
    max_detections=0,  # Strictly no bees
    search_limit=500   # Search up to 500 frames
)

print(f"\n✓ Blob detector initialized")
print(f"  Clean frames used: {num_clean}")

# Visualize background model
bg_img = blob.get_background_image()
if bg_img is not None:
    show_frame(bg_img, "Background Model")

# Test on first frame
print("\nTesting blob detector on first frame...")
blob_dets = blob.detect(first_frame)
print(f"  Detections: {len(blob_dets)}")

if blob_dets:
    frame_with_blobs = draw_detections(first_frame, blob_dets, color=(0, 255, 0))
    show_frame(frame_with_blobs, f"Blob Detections (n={len(blob_dets)})")
    
    # Show background difference
    diff_viz = blob.visualize_background_difference(first_frame)
    show_frame(diff_viz, "Background Difference Visualization")

## 1.5: Initialize SIFT Detector (DL-Guided)

In [ ]:
print("Initializing SIFT Detector (DL-guided template learning)...")
print("This learns bee appearance patterns from YOLO detections.")
print("May take a few minutes...\n")

sift = SIFTDetector(
    min_keypoints=3,
    cluster_eps=30.0
)

# DL-guided template learning
num_templates = sift.initialize_from_video(
    video_path=VIDEO_PATH,
    yolo_detector=yolo,
    num_frames=100,  # Learn from 100 frames
    start_frame=200,  # After background initialization
    min_confidence=0.7  # High confidence only
)

print(f"\n✓ SIFT detector initialized")
print(f"  Templates learned: {num_templates}")

# Save templates
sift.save_templates(str(TEMPLATES_PATH))
print(f"  Templates saved to: {TEMPLATES_PATH}")

# Test on first frame
print("\nTesting SIFT detector on first frame...")
sift_dets = sift.detect(first_frame, use_templates=True)
print(f"  Detections: {len(sift_dets)}")

if sift_dets:
    frame_with_sift = draw_detections(first_frame, sift_dets, color=(0, 0, 255))
    show_frame(frame_with_sift, f"SIFT Detections (n={len(sift_dets)})")

## 1.6: Initialize Noise Filter (Optional)

In [ ]:
noise_filter = None

if USE_NOISE_FILTER and CNN_MODEL and NoiseFilter is not None:
    print("Initializing CNN noise filter...")
    
    try:
        noise_filter = NoiseFilter(
            model_path=CNN_MODEL,
            noise_threshold=0.9,
            batch_size=32
        )
        
        print(f"✓ Noise filter loaded")
        print(f"  Model: {CNN_MODEL}")
        if hasattr(noise_filter, 'device'):
            print(f"  Device: {noise_filter.device}")
        if hasattr(noise_filter, 'noise_threshold'):
            print(f"  Threshold: {noise_filter.noise_threshold}")
        
        # Test filtering
        if blob_dets:
            print("\nTesting noise filter...")
            filtered = noise_filter.filter_detections(first_frame, blob_dets)
            print(f"  Before: {len(blob_dets)} detections")
            print(f"  After: {len(filtered)} detections")
            print(f"  Removed: {len(blob_dets) - len(filtered)} false positives")
    except Exception as e:
        print(f"⚠ Failed to load noise filter: {e}")
        noise_filter = None
else:
    if NoiseFilter is None:
        print("Noise filter not available (import failed)")
    else:
        print("Noise filter disabled")

## 1.7: Combined Detection Test

In [ ]:
print("Testing combined detection pipeline...\n")

# Get detections from all sources
all_blob = blob.detect(first_frame)
all_sift = sift.detect(first_frame, use_templates=True)
all_yolo = yolo.detect(first_frame)

print(f"Detection counts:")
print(f"  Blob: {len(all_blob)}")
print(f"  SIFT: {len(all_sift)}")
print(f"  YOLO: {len(all_yolo)}")

# Combine
combined = all_blob + all_sift + all_yolo

# Apply noise filter if available
if noise_filter:
    combined = noise_filter.filter_detections(first_frame, combined)
    print(f"  Combined (filtered): {len(combined)}")
else:
    print(f"  Combined: {len(combined)}")

# Visualize all detections
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Original
axes[0, 0].imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Original Frame')
axes[0, 0].axis('off')

# Blob
blob_viz = draw_detections(first_frame, all_blob, (0, 255, 0))
axes[0, 1].imshow(cv2.cvtColor(blob_viz, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title(f'Blob Detections (n={len(all_blob)})')
axes[0, 1].axis('off')

# SIFT
sift_viz = draw_detections(first_frame, all_sift, (0, 0, 255))
axes[1, 0].imshow(cv2.cvtColor(sift_viz, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title(f'SIFT Detections (n={len(all_sift)})')
axes[1, 0].axis('off')

# Combined
combined_viz = draw_detections(first_frame, combined, (255, 0, 255))
axes[1, 1].imshow(cv2.cvtColor(combined_viz, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title(f'Combined Detections (n={len(combined)})')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\n✓ Phase 1 complete - All detectors initialized")

---
# Phase 2: Tracking

Create tracking system and process video.

## 2.1: Create Tracking System

In [ ]:
print("Creating tracking system...\n")

# Create MOT algorithm
mot = BeeTracker(
    config=config,
    tracking_classes=['bee', 'wasp']
)
print(f"✓ MOT algorithm: BeeTracker")

# Create high-level tracking system
tracker = BeeTracking(
    mot_algorithm=mot,
    yolo_model=yolo_model,
    detection_mode=DETECTION_MODE,
    use_noise_filter=USE_NOISE_FILTER,
    noise_filter_model=noise_filter,
    config=config
)
print(f"✓ Tracking system created")
print(f"  Detection mode: {DETECTION_MODE.value}")
print(f"  Noise filter: {USE_NOISE_FILTER}")

# Use initialized detectors
tracker.blob_detector = blob
tracker.sift_detector = sift
print(f"✓ Detectors attached")
print(f"  Blob: initialized with {num_clean} frames")
print(f"  SIFT: initialized with {num_templates} templates")

print("\n✓ Tracking system ready")

## 2.2: Test Single Frame Tracking

In [ ]:
print("Testing single frame tracking...\n")

# Process first frame
result = tracker.process_frame(first_frame, frame_num=0)

detections = result['detections']
tracks = result['tracks']
mode = result['mode']

print(f"Results:")
print(f"  Detections: {len(detections)}")
print(f"  Tracks: {len(tracks)}")
print(f"  Mode: {mode}")

# Visualize tracks
if tracks:
    frame_with_tracks = draw_tracks(first_frame, tracks)
    show_frame(frame_with_tracks, f"Tracks (n={len(tracks)})")
    
    # Show track details
    print("\nTrack details:")
    for track_id, track in list(tracks.items())[:5]:  # First 5
        print(f"  Track {track_id}:")
        print(f"    Position: {track.centroid}")
        print(f"    Velocity: {track.velocity}")
        print(f"    Age: {track.age}")
        print(f"    Label: {track.label}")

# Reset for full video processing
tracker.reset()
print("\n✓ Single frame test complete (tracker reset)")

## 2.3: Process Full Video

In [ ]:
print("Processing full video...")
print(f"This may take several minutes for {total_frames} frames.\n")

# Process video with progress bar
results = tracker.process_video(
    video_path=VIDEO_PATH,
    roi=ROI
)

print(f"\n✓ Video processing complete")
print(f"  Total frames: {results['frame'].max() + 1}")
print(f"  Total tracks: {results['track_id'].nunique()}")
print(f"  Total detections: {len(results)}")

# Get statistics
stats = tracker.get_statistics()
print(f"\nTracking statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

# Save results
results.to_csv(TRACKING_CSV, index=False)
print(f"\n✓ Results saved to: {TRACKING_CSV}")

# Display sample
print(f"\nSample results:")
display(results.head(10))

## 2.4: Visualize Tracking Statistics

In [ ]:
print("Generating tracking statistics visualizations...\n")

plot_tracking_stats(results)

# Additional statistics
print("Track Statistics:")
track_lengths = results.groupby('track_id').size()
print(f"  Total unique tracks: {len(track_lengths)}")
print(f"  Mean track length: {track_lengths.mean():.1f} frames")
print(f"  Median track length: {track_lengths.median():.1f} frames")
print(f"  Max track length: {track_lengths.max()} frames")
print(f"  Min track length: {track_lengths.min()} frames")

# Long tracks
long_tracks = track_lengths[track_lengths > 100]
if len(long_tracks) > 0:
    print(f"\n  Tracks > 100 frames: {len(long_tracks)}")
    print(f"  Longest 5 tracks: {sorted(long_tracks.values, reverse=True)[:5]}")

## 2.5: Visualize Sample Trajectories

In [ ]:
print("Visualizing sample trajectories...\n")

# Get longest tracks
track_lengths = results.groupby('track_id').size()
longest_tracks = track_lengths.nlargest(5).index

# Plot trajectories
plt.figure(figsize=(12, 8))

for track_id in longest_tracks:
    track_data = results[results['track_id'] == track_id]
    
    # Calculate centroids
    cx = (track_data['x1'] + track_data['x2']) / 2
    cy = (track_data['y1'] + track_data['y2']) / 2
    
    plt.plot(cx, cy, marker='o', markersize=2, alpha=0.7,
            label=f'Track {track_id} ({len(track_data)} frames)')
    
    # Mark start and end
    plt.scatter(cx.iloc[0], cy.iloc[0], s=100, marker='o', 
               edgecolors='black', linewidths=2, zorder=10)
    plt.scatter(cx.iloc[-1], cy.iloc[-1], s=100, marker='x',
               linewidths=2, zorder=10)

plt.xlabel('X coordinate')
plt.ylabel('Y coordinate')
plt.title('Sample Bee Trajectories (○ = start, × = end)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Showing trajectories for {len(longest_tracks)} longest tracks")

---
# Phase 3: Event Analysis

Detect entry/exit events (if EventProcessor is implemented).

In [ ]:
# Placeholder for event analysis
# Uncomment when EventProcessor is implemented

# try:
#     from beemonitor.processing import EventProcessor
#     
#     print("Processing events...\n")
#     
#     # Use already-detected nests from Phase 1
#     print(f"Using {len(nests)} nests detected in Phase 1")
#     
#     # Process events
#     event_processor = EventProcessor(nests)
#     events = event_processor.process_tracks(results)
#     
#     print(f"\n✓ Event analysis complete")
#     print(f"  Total events: {len(events)}")
#     
#     # Save events
#     events_df = pd.DataFrame(events)
#     events_df.to_csv(EVENTS_CSV, index=False)
#     print(f"  Saved to: {EVENTS_CSV}")
#     
#     # Display sample
#     print("\nSample events:")
#     display(events_df.head(10))
#     
# except ImportError:
#     print("EventProcessor not available - skipping event analysis")

print("Event analysis placeholder - implement when EventProcessor is ready")

---
# Phase 4: Visualization

Create annotated output video.

## 4.1: Create Visualization Video

In [ ]:
print("Creating visualization video...")
print("This may take several minutes.\n")

cap = cv2.VideoCapture(VIDEO_PATH)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(str(VIZ_VIDEO), fourcc, fps, (width, height))

frame_num = 0
pbar = tqdm(total=total_frames, desc="Rendering")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Get tracks for this frame
    frame_tracks = results[results['frame'] == frame_num]
    
    # Draw tracks
    for _, track in frame_tracks.iterrows():
        track_id = int(track['track_id'])
        x1, y1, x2, y2 = int(track['x1']), int(track['y1']), int(track['x2']), int(track['y2'])
        
        # Color by track ID
        color_idx = track_id % 20
        color = tuple(int(c * 255) for c in plt.cm.tab20(color_idx / 20)[:3])
        
        # Draw bbox and ID
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"ID:{track_id}", (x1, y1-10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    # Add frame info
    info_text = f"Frame: {frame_num}/{total_frames} | Tracks: {len(frame_tracks)}"
    cv2.putText(frame, info_text, (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    out.write(frame)
    frame_num += 1
    pbar.update(1)

cap.release()
out.release()
pbar.close()

print(f"\n✓ Visualization video created: {VIZ_VIDEO}")
print(f"  Resolution: {width}x{height}")
print(f"  FPS: {fps}")
print(f"  Frames: {total_frames}")

## 4.2: Display Sample Frames with Tracks

In [ ]:
print("Displaying sample frames with tracks...\n")

# Select sample frames (evenly distributed)
sample_frames = np.linspace(0, total_frames-1, 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

cap = cv2.VideoCapture(VIDEO_PATH)

for idx, frame_num in enumerate(sample_frames):
    # Read frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    
    if not ret:
        continue
    
    # Get tracks
    frame_tracks = results[results['frame'] == frame_num]
    
    # Draw tracks
    for _, track in frame_tracks.iterrows():
        track_id = int(track['track_id'])
        x1, y1, x2, y2 = int(track['x1']), int(track['y1']), int(track['x2']), int(track['y2'])
        
        color_idx = track_id % 20
        color = tuple(int(c * 255) for c in plt.cm.tab20(color_idx / 20)[:3])
        
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"{track_id}", (x1, y1-5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # Display
    axes[idx].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f'Frame {frame_num} ({len(frame_tracks)} tracks)')
    axes[idx].axis('off')

cap.release()
plt.tight_layout()
plt.show()

print("✓ Sample frames displayed")

---
# Summary

In [ ]:
print("="*60)
print("BEEMONITOR PIPELINE TEST COMPLETE")
print("="*60)

print("\nInput:")
print(f"  Video: {VIDEO_PATH}")
print(f"  Frames: {total_frames}")
print(f"  Duration: {total_frames/fps:.1f}s")
print(f"  Resolution: {width}x{height}")

print("\nPhase 1: Initialization")
print(f"  ✓ Nest detection: {len(nests)} nests")
print(f"  ✓ ROI: {ROI}")
print(f"  ✓ Blob detector: {num_clean} clean frames")
print(f"  ✓ SIFT detector: {num_templates} templates")
print(f"  ✓ YOLO detector: {yolo.device}")
if noise_filter:
    print(f"  ✓ Noise filter: {noise_filter.device}")

print("\nPhase 2: Tracking")
print(f"  ✓ Detection mode: {DETECTION_MODE.value}")
print(f"  ✓ Total tracks: {results['track_id'].nunique()}")
print(f"  ✓ Total detections: {len(results)}")
print(f"  ✓ Mean track length: {results.groupby('track_id').size().mean():.1f} frames")

print("\nOutputs:")
print(f"  ✓ Tracking results: {TRACKING_CSV}")
print(f"  ✓ SIFT templates: {TEMPLATES_PATH}")
print(f"  ✓ Visualization video: {VIZ_VIDEO}")

print("\n" + "="*60)
print("All tests passed! System is ready for production use.")
print("="*60)